# Metaclasses — Advanced Problems with Solutions
## Tutorial-style edition

This notebook takes the same topic — **Python metaclasses** — but approaches it as a guided problem-solving tutorial.

Instead of presenting large finished examples immediately, we will repeatedly do this:

1. state a concrete problem,
2. reason about *where* in Python's class-creation process the problem belongs,
3. build the solution in small pieces,
4. inspect intermediate results,
5. assemble the final implementation,
6. test edge cases,
7. discuss when a metaclass is — and is not — the right tool.

The examples are intentionally advanced, but each one is broken into small logical steps.


### A quick reminder

A class object is itself an object.

For ordinary user-defined classes, the object that creates the class is normally `type`.

So:

```python
class Person:
    pass
```

is conceptually related to:

```python
Person = type("Person", (), namespace)
```

The class statement is much more convenient because Python builds the namespace, executes the body, resolves bases, chooses the metaclass, and then asks that metaclass to create the class.


In [1]:
class Person:
    pass

print(type(Person))
print(isinstance(Person, type))

assert type(Person) is type
assert isinstance(Person, type)


<class 'type'>
True


The important idea for the rest of the notebook is:

> A metaclass customizes the creation or behavior of **class objects**.

That gives us several possible intervention points:

- `__prepare__` — choose/customize the namespace used while the class body executes,
- `__new__` — create or transform the class object,
- `__init__` — initialize the newly created class object,
- `__call__` — control what happens when the class is later called to create instances.

We will use all of them, but only when the problem actually needs them.


# Problem 1 — Predict which metaclass Python will choose

Before writing custom metaclasses, we need to understand how Python chooses one.

Suppose a base class already has a custom metaclass.

What happens when we subclass it without explicitly specifying `metaclass=`?

And what happens when we try to force an incompatible metaclass?


### Step 1 — Create a custom metaclass

We will make a minimal metaclass that marks every class it creates.


In [2]:
class MarkingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.created_by_marking_meta = True
        return cls


### Step 2 — Use it on a base class

The interesting part is not just the base class.

We want to see what happens to a subclass that does **not** repeat `metaclass=MarkingMeta`.


In [3]:
class MarkedBase(metaclass=MarkingMeta):
    pass


class Child(MarkedBase):
    pass


print(type(MarkedBase))
print(type(Child))

assert type(MarkedBase) is MarkingMeta
assert type(Child) is MarkingMeta
assert Child.created_by_marking_meta is True


<class '__main__.MarkingMeta'>
<class '__main__.MarkingMeta'>


The subclass automatically uses a compatible metaclass.

This is necessary because the metaclass of a derived class must be compatible with the metaclasses of all its bases.

In simplified terms, Python needs a metaclass that is a subclass of the metaclasses required by the base classes.


### Step 3 — Trigger a conflict

Now we create a second, unrelated metaclass.

Neither custom metaclass is a subclass of the other.


In [4]:
class OtherMeta(type):
    pass


class OtherBase(metaclass=OtherMeta):
    pass


try:
    class BrokenCombination(MarkedBase, OtherBase):
        pass
except TypeError as exc:
    print("Expected metaclass conflict:")
    print(exc)
else:
    raise AssertionError("A metaclass conflict should have occurred")


Expected metaclass conflict:
metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases


### Step 4 — Resolve the conflict

The usual solution is to create a metaclass that inherits from both required metaclasses.

This only works cleanly when the metaclasses themselves use cooperative `super()`.


In [5]:
class CombinedMeta(MarkingMeta, OtherMeta):
    pass


class WorkingCombination(
    MarkedBase,
    OtherBase,
    metaclass=CombinedMeta
):
    pass


assert type(WorkingCombination) is CombinedMeta
assert WorkingCombination.created_by_marking_meta is True

print(type(WorkingCombination))


<class '__main__.CombinedMeta'>


### Solution takeaway

The metaclass of a derived class is constrained by the metaclasses of its bases.

This matters in real frameworks because libraries may already use metaclasses internally. A custom metaclass can therefore create compatibility problems even when your own code looks simple.


# Problem 2 — Observe class-body assignments with `__prepare__`

Suppose we are building a declarative framework and want to know the exact order in which public attributes were assigned inside a class body.

A normal metaclass `__new__` only sees the namespace **after** the body has executed.

If we want to observe assignments *while they happen*, we need `__prepare__`.


### Step 1 — Build a namespace object that records assignments

A class namespace only needs to behave like a mapping.

We can subclass `dict` and record keys whenever `__setitem__` is called.


In [6]:
class TrackingNamespace(dict):
    def __init__(self):
        super().__init__()
        self.assignment_order = []

    def __setitem__(self, key, value):
        if not key.startswith("__"):
            self.assignment_order.append(key)
        super().__setitem__(key, value)


### Step 2 — Return that mapping from `__prepare__`

`__prepare__` is called before the class body executes.

The returned object becomes the namespace into which class-body assignments are written.


In [7]:
class TrackingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return TrackingNamespace()

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, dict(namespace))
        cls.__assignment_order__ = tuple(namespace.assignment_order)
        return cls


### Step 3 — Try it

Notice that methods are assignments too: the function object is created and then stored under the method name in the class namespace.


In [8]:
class TutorialExample(metaclass=TrackingMeta):
    first = 1
    second = 2

    def method_a(self):
        return "A"

    third = 3

    def method_b(self):
        return "B"


print(TutorialExample.__assignment_order__)

assert TutorialExample.__assignment_order__ == (
    "first",
    "second",
    "method_a",
    "third",
    "method_b",
)


('first', 'second', 'method_a', 'third', 'method_b')


### Step 4 — Why not just inspect `vars(cls)`?

Modern Python dictionaries preserve insertion order, so for many use cases you can inspect the final namespace and get declaration order.

`__prepare__` becomes useful when you need behavior that a normal dictionary does not provide, such as:

- rejecting duplicate writes,
- preserving richer metadata,
- transforming names as they are assigned,
- recording assignment events,
- using a specialized mapping object.

Do not add `__prepare__` merely because metaclasses support it.


# Problem 3 — Validate annotated class fields at class creation time

Imagine a framework in which a class defines configuration fields using annotations:

```python
class RetryPolicy:
    attempts: int = 3
    backoff: float = 0.5
```

We want a metaclass to verify that:

- every public annotated field has a default value,
- the default has the expected runtime type,
- boolean values do not accidentally count as integers,
- unsupported annotation forms produce a clear error.

This is a class-definition rule, so class creation is a reasonable place to enforce it.


### Step 1 — Inspect `__annotations__`

Annotations written in a class body are stored in the namespace under `__annotations__`.


In [9]:
class AnnotationDemo:
    attempts: int = 3
    backoff: float = 0.5

print(AnnotationDemo.__annotations__)

assert AnnotationDemo.__annotations__ == {
    "attempts": int,
    "backoff": float,
}


{'attempts': <class 'int'>, 'backoff': <class 'float'>}


### Step 2 — Write one reusable validator

We will deliberately support only a small set of runtime-checkable types.

A production framework would need a much richer strategy for things such as unions, generics, forward references, and `typing.Annotated`.


In [10]:
def value_matches_annotation(value, annotation):
    if annotation is int:
        return isinstance(value, int) and not isinstance(value, bool)

    if annotation is float:
        return (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
        )

    if annotation is str:
        return isinstance(value, str)

    if annotation is bool:
        return isinstance(value, bool)

    raise TypeError(
        f"Unsupported annotation for this tutorial: {annotation!r}"
    )


### Step 3 — Move the rule into a metaclass

The metaclass checks only fields declared directly in the current class body.

That keeps the example focused. In a larger framework we could also merge inherited schema metadata.


In [11]:
class TypedDefaultsMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        annotations = namespace.get("__annotations__", {})

        for field_name, annotation in annotations.items():
            if field_name.startswith("_"):
                continue

            if field_name not in namespace:
                raise TypeError(
                    f"{name}.{field_name} must define a default value"
                )

            value = namespace[field_name]

            if not value_matches_annotation(value, annotation):
                raise TypeError(
                    f"{name}.{field_name}={value!r} does not match "
                    f"{annotation!r}"
                )

        return super().__new__(mcls, name, bases, namespace)


### Step 4 — Test a valid class


In [12]:
class RetryPolicy(metaclass=TypedDefaultsMeta):
    attempts: int = 3
    backoff: float = 0.5
    label: str = "standard"
    enabled: bool = True


assert RetryPolicy.attempts == 3
assert RetryPolicy.backoff == 0.5


### Step 5 — Test two invalid definitions

The failure happens while the class statement executes, before the invalid class can be used.


In [13]:
try:
    class MissingDefault(metaclass=TypedDefaultsMeta):
        attempts: int
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Missing default should have failed")


try:
    class WrongDefault(metaclass=TypedDefaultsMeta):
        attempts: int = True
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("bool should not be accepted as int here")


MissingDefault.attempts must define a default value
WrongDefault.attempts=True does not match <class 'int'>


### Solution takeaway

Metaclasses are good at enforcing rules about **class definitions**.

But runtime type validation for ordinary application data is often better handled by normal constructors, descriptors, dataclasses, or dedicated validation libraries.


# Problem 4 — Preserve `__classcell__` when transforming the namespace

This is a subtle metaclass problem.

A method that uses zero-argument `super()` or references `__class__` may require a hidden namespace entry named `__classcell__`.

If a metaclass reconstructs the namespace carelessly and drops this entry, class creation can fail.


### Step 1 — See a normal class using zero-argument `super()`


In [14]:
class Parent:
    def message(self):
        return "parent"


class NormalChild(Parent):
    def message(self):
        return super().message() + " -> child"


assert NormalChild().message() == "parent -> child"


### Step 2 — Write a safe transforming metaclass

A common best practice is to modify or copy the namespace without accidentally dropping special entries.

Here we uppercase only selected string constants and preserve everything else.


In [15]:
class UpperConstantsMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        transformed = dict(namespace)

        for key, value in list(transformed.items()):
            if (
                key.isupper()
                and isinstance(value, str)
            ):
                transformed[key] = value.upper()

        # Important: transformed still contains __classcell__ if Python created it.
        return super().__new__(mcls, name, bases, transformed)


### Step 3 — Confirm that `super()` still works


In [16]:
class SafeChild(Parent, metaclass=UpperConstantsMeta):
    LABEL = "child label"

    def message(self):
        return super().message() + " -> safe child"


assert SafeChild.LABEL == "CHILD LABEL"
assert SafeChild().message() == "parent -> safe child"

print(SafeChild.LABEL)
print(SafeChild().message())


CHILD LABEL
parent -> safe child


### Step 4 — Understand the failure mode

We will not intentionally break the notebook by constructing a malformed class, but the dangerous pattern looks like this conceptually:

```python
clean_namespace = {
    key: value
    for key, value in namespace.items()
    if not key.startswith("__")
}
```

That can discard `__classcell__`.

The lesson is broader than this one attribute:

> When rebuilding a class namespace, preserve Python's special class-construction entries unless you have a very specific reason not to.


# Problem 5 — Build descriptors first, then let a metaclass assemble them

A common mistake is to put too much logic directly into the metaclass.

Suppose we want fields with validation.

The **field itself** knows how to validate one value, so that behavior belongs naturally in a descriptor.

The metaclass should focus on class-level assembly.


### Step 1 — Build a reusable descriptor


In [17]:
class ValidatedField:
    def __init__(self, validator):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if not self.validator(value):
            raise ValueError(
                f"Invalid value for {self.name}: {value!r}"
            )
        setattr(instance, self.storage_name, value)


### Step 2 — Create a few field factories

Keeping these factories tiny makes class declarations readable.


In [18]:
def positive_int():
    return ValidatedField(
        lambda value: (
            isinstance(value, int)
            and not isinstance(value, bool)
            and value > 0
        )
    )


def non_empty_string():
    return ValidatedField(
        lambda value: isinstance(value, str) and bool(value.strip())
    )


### Step 3 — Let the metaclass collect the descriptors

The metaclass does not perform value validation.

It simply discovers fields and stores class-level metadata.


In [19]:
class ValidatedModelMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        inherited = {}

        for base in bases:
            inherited.update(getattr(base, "__validated_fields__", {}))

        local = {
            key: value
            for key, value in namespace.items()
            if isinstance(value, ValidatedField)
        }

        inherited.update(local)
        namespace["__validated_fields__"] = inherited

        return super().__new__(mcls, name, bases, namespace)


### Step 4 — Use the framework


In [20]:
class UserRecord(metaclass=ValidatedModelMeta):
    user_id = positive_int()
    username = non_empty_string()

    def __init__(self, user_id, username):
        self.user_id = user_id
        self.username = username


user = UserRecord(10, "ada")

assert user.user_id == 10
assert user.username == "ada"
assert set(UserRecord.__validated_fields__) == {
    "user_id",
    "username",
}


### Step 5 — Check validation

Because validation lives in the descriptor, it also runs on later assignments — not only during construction.


In [21]:
try:
    user.user_id = 0
except ValueError as exc:
    print(exc)
else:
    raise AssertionError("0 should have been rejected")


try:
    user.username = "   "
except ValueError as exc:
    print(exc)
else:
    raise AssertionError("Blank username should have been rejected")


Invalid value for user_id: 0
Invalid value for username: '   '


### Solution takeaway

This division of responsibility is usually healthier:

- descriptor → one attribute's runtime behavior,
- metaclass → class-wide discovery and metadata.

A metaclass should not become a dumping ground for every framework feature.


# Problem 6 — Wrap instance, static, and class methods correctly

Suppose a framework wants to record how often public methods are called.

A naive metaclass might check `callable(value)` and wrap everything.

That is not enough because `staticmethod` and `classmethod` are descriptor objects inside the class namespace.

We need to handle each method kind deliberately.


### Step 1 — Inspect what the class namespace actually contains


In [22]:
class MethodKinds:
    def instance_method(self):
        pass

    @staticmethod
    def static_method():
        pass

    @classmethod
    def class_method(cls):
        pass


raw = vars(MethodKinds)

print(type(raw["instance_method"]))
print(type(raw["static_method"]))
print(type(raw["class_method"]))


<class 'function'>
<class 'staticmethod'>
<class 'classmethod'>


The instance method is stored as a function.

The static and class methods are stored as descriptor wrappers.

To preserve their semantics, we should unwrap their underlying functions, wrap those functions, and then rebuild the correct descriptor.


### Step 2 — Create a wrapper factory

Using a factory avoids a late-binding closure bug when looping over many methods.


In [23]:
from functools import wraps


def make_counted_wrapper(method_name, func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        owner = None

        if args:
            first = args[0]
            if isinstance(first, type):
                owner = first
            else:
                owner = type(first)

        if owner is not None and hasattr(owner, "__method_calls__"):
            owner.__method_calls__[method_name] += 1

        return func(*args, **kwargs)

    return wrapper


### Step 3 — Build the metaclass

We will:

- wrap ordinary public functions,
- unwrap and rebuild `staticmethod`,
- unwrap and rebuild `classmethod`,
- leave properties alone,
- store counters on the class.


In [24]:
import inspect


class CountingMethodsMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        transformed = dict(namespace)
        counters = {}

        for attr_name, value in namespace.items():
            if attr_name.startswith("_"):
                continue

            if inspect.isfunction(value):
                counters[attr_name] = 0
                transformed[attr_name] = make_counted_wrapper(
                    attr_name,
                    value,
                )

            elif isinstance(value, staticmethod):
                counters[attr_name] = 0
                wrapped = make_counted_wrapper(
                    attr_name,
                    value.__func__,
                )
                transformed[attr_name] = staticmethod(wrapped)

            elif isinstance(value, classmethod):
                counters[attr_name] = 0
                wrapped = make_counted_wrapper(
                    attr_name,
                    value.__func__,
                )
                transformed[attr_name] = classmethod(wrapped)

        transformed["__method_calls__"] = counters

        return super().__new__(mcls, name, bases, transformed)


### Step 4 — Exercise all three method kinds

There is one deliberate design choice here:

A static method has no implicit class or instance argument, so our generic wrapper cannot infer the owning class from its arguments. Therefore its counter will remain unchanged in this implementation.

That is not a bug in Python — it is a limitation of this particular wrapper strategy.

A production design might close over the class after creation or use a different instrumentation mechanism.


In [25]:
class Utility(metaclass=CountingMethodsMeta):
    def double(self, value):
        return value * 2

    @staticmethod
    def add(a, b):
        return a + b

    @classmethod
    def class_name(cls):
        return cls.__name__


u = Utility()

assert u.double(5) == 10
assert Utility.add(2, 3) == 5
assert Utility.class_name() == "Utility"

assert Utility.__method_calls__["double"] == 1
assert Utility.__method_calls__["class_name"] == 1
assert Utility.__method_calls__["add"] == 0

print(Utility.__method_calls__)


{'double': 1, 'add': 0, 'class_name': 1}


### Solution takeaway

When transforming class attributes, work with the actual objects stored in the namespace.

Functions, static methods, class methods, and properties participate in Python's descriptor protocol differently.


# Problem 7 — Customize `isinstance` and `issubclass`

Metaclasses can also customize class-level operators.

Two particularly advanced hooks are:

- `__instancecheck__`
- `__subclasscheck__`

We can use them to build a tiny structural interface: any class that provides certain methods can count as compatible, even without explicit inheritance.


### Step 1 — Define the required method names

We will create a "JSON-like serializer" interface.

A compatible class must provide both `dumps` and `loads`.


In [26]:
class StructuralMeta(type):
    def __subclasscheck__(cls, candidate):
        required = getattr(cls, "__required_methods__", ())

        return all(
            callable(getattr(candidate, name, None))
            for name in required
        )

    def __instancecheck__(cls, instance):
        return issubclass(type(instance), cls)


### Step 2 — Create the structural interface class


In [27]:
class SerializerLike(metaclass=StructuralMeta):
    __required_methods__ = ("dumps", "loads")


### Step 3 — Create a class that never inherits from it


In [28]:
class SimpleSerializer:
    def dumps(self, obj):
        return repr(obj)

    def loads(self, text):
        return text


serializer = SimpleSerializer()

assert issubclass(SimpleSerializer, SerializerLike)
assert isinstance(serializer, SerializerLike)

print(issubclass(SimpleSerializer, SerializerLike))
print(isinstance(serializer, SerializerLike))


True
True


### Step 4 — Reject an incomplete class


In [29]:
class IncompleteSerializer:
    def dumps(self, obj):
        return repr(obj)


assert not issubclass(IncompleteSerializer, SerializerLike)
assert not isinstance(IncompleteSerializer(), SerializerLike)


### Solution takeaway

This is educational, but modern Python often uses `typing.Protocol` for structural typing because it works better with static type checkers and makes intent clearer.

Customizing `isinstance` and `issubclass` can surprise readers, so use it only when the semantics are well documented.


# Problem 8 — Control instance creation with metaclass `__call__`

Calling a class normally means:

```python
obj = MyClass(...)
```

But the class itself is an instance of its metaclass.

So the operation is routed through the metaclass's `__call__`.

We can use that to normalize constructor arguments and cache instances.


### Step 1 — Define the desired behavior

We want:

```python
Tag(" Python ") is Tag("python")
```

The constructor argument should be normalized before cache lookup.


### Step 2 — Put normalization on the class

The metaclass should not know domain-specific rules such as how tags are normalized.

Instead, the class exposes a small hook named `_cache_key`.


In [30]:
class NormalizedCacheMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.__cache__ = {}
        return cls

    def __call__(cls, *args, **kwargs):
        key = cls._cache_key(*args, **kwargs)

        if key in cls.__cache__:
            return cls.__cache__[key]

        instance = super().__call__(*args, **kwargs)
        cls.__cache__[key] = instance
        return instance


### Step 3 — Define a cache-aware class

The object's stored value should use the same normalization rule as the cache key.


In [31]:
class Tag(metaclass=NormalizedCacheMeta):
    @staticmethod
    def _normalize(text):
        return text.strip().lower()

    @classmethod
    def _cache_key(cls, text):
        return cls._normalize(text)

    def __init__(self, text):
        self.text = self._normalize(text)


a = Tag(" Python ")
b = Tag("python")
c = Tag("PYTHON")
d = Tag("metaclasses")

assert a is b is c
assert a is not d
assert a.text == "python"

print(id(a), id(b), id(c), id(d))


1356491206544 1356491206544 1356491206544 1356491835344


### Step 4 — Recognize the trade-off

This looks convenient, but it changes a very fundamental expectation: calling a class does not necessarily create a new object.

That can affect tests, identity-sensitive code, resource ownership, and subclass behavior.

A factory function such as `get_tag("python")` is often more explicit.


# Problem 9 — Combine an abstract base class with automatic registration

We want a command framework with these rules:

- the root class is abstract,
- intermediate abstract subclasses are allowed,
- concrete subclasses register automatically,
- each concrete class declares a unique command name.

Because abstract base classes already use `ABCMeta`, our registry metaclass should extend `ABCMeta` rather than plain `type`.


### Step 1 — Build the registry metaclass


In [32]:
import inspect
from abc import ABC, ABCMeta, abstractmethod


class CommandMeta(ABCMeta):
    registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)

        if not inspect.isabstract(cls):
            command_name = namespace.get("command_name")

            if not isinstance(command_name, str) or not command_name:
                raise TypeError(
                    f"Concrete command {name} must define command_name"
                )

            if command_name in mcls.registry:
                raise TypeError(
                    f"Duplicate command name: {command_name!r}"
                )

            mcls.registry[command_name] = cls

        return cls


### Step 2 — Define the abstract API

The abstract method keeps incomplete subclasses from being treated as concrete commands.


In [33]:
class Command(ABC, metaclass=CommandMeta):
    @abstractmethod
    def execute(self, context):
        raise NotImplementedError


### Step 3 — Add an abstract intermediate class

It should *not* be registered because it still has an abstract method.


In [34]:
class AuthenticatedCommand(Command):
    @abstractmethod
    def authorize(self, context):
        raise NotImplementedError

    def execute(self, context):
        self.authorize(context)
        return self.run(context)

    @abstractmethod
    def run(self, context):
        raise NotImplementedError


assert "AuthenticatedCommand" not in CommandMeta.registry


### Step 4 — Add concrete implementations


In [35]:
class PingCommand(Command):
    command_name = "ping"

    def execute(self, context):
        return "pong"


class AdminStatusCommand(AuthenticatedCommand):
    command_name = "admin-status"

    def authorize(self, context):
        if not context.get("is_admin"):
            raise PermissionError("Admin access required")

    def run(self, context):
        return "all systems operational"


assert CommandMeta.registry["ping"] is PingCommand
assert CommandMeta.registry["admin-status"] is AdminStatusCommand

print(sorted(CommandMeta.registry))


['admin-status', 'ping']


### Step 5 — Use the registry as a factory


In [36]:
def make_command(name):
    command_class = CommandMeta.registry[name]
    return command_class()


assert make_command("ping").execute({}) == "pong"

admin = make_command("admin-status")
assert admin.execute({"is_admin": True}) == "all systems operational"


### Solution takeaway

When extending framework metaclasses such as `ABCMeta`, inherit from the framework metaclass and keep using `super()`.

That preserves the existing framework behavior instead of replacing it accidentally.


# Problem 10 — Design a clean class-header configuration API

Class statements can contain keyword arguments:

```python
class User(Resource, endpoint="/users", version=2):
    ...
```

Those keywords are passed into the class-creation machinery.

We will build a metaclass that consumes them, validates them, and stores normalized configuration.


### Step 1 — Decide which class is allowed to omit configuration

Our root framework class should be allowed to exist without an endpoint.

Concrete subclasses must provide one.


In [37]:
class ResourceMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        endpoint=None,
        version=1,
        abstract=False,
        **kwargs,
    ):
        if kwargs:
            unknown = ", ".join(sorted(kwargs))
            raise TypeError(
                f"Unknown class option(s) for {name}: {unknown}"
            )

        if not abstract:
            if not isinstance(endpoint, str) or not endpoint.startswith("/"):
                raise TypeError(
                    f"{name} must define endpoint='/...'"
                )

            if (
                not isinstance(version, int)
                or isinstance(version, bool)
                or version < 1
            ):
                raise TypeError(
                    f"{name} version must be a positive integer"
                )

        namespace["__endpoint__"] = endpoint
        namespace["__version__"] = version
        namespace["__abstract_resource__"] = abstract

        return super().__new__(mcls, name, bases, namespace)


### Step 2 — Declare the root and concrete resources


In [38]:
class Resource(
    metaclass=ResourceMeta,
    abstract=True,
):
    pass


class UsersResource(
    Resource,
    endpoint="/users",
    version=2,
):
    pass


class OrdersResource(
    Resource,
    endpoint="/orders",
):
    pass


assert UsersResource.__endpoint__ == "/users"
assert UsersResource.__version__ == 2
assert OrdersResource.__version__ == 1


### Step 3 — Verify failures are clear


In [39]:
try:
    class MissingEndpoint(Resource):
        pass
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Missing endpoint should fail")


try:
    class BadVersion(
        Resource,
        endpoint="/bad",
        version=0,
    ):
        pass
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Bad version should fail")


MissingEndpoint must define endpoint='/...'
BadVersion version must be a positive integer


### Solution takeaway

Class-header keywords can make declarative APIs elegant.

But the metaclass should:

- validate them explicitly,
- consume only the options it understands,
- reject misspelled or unknown options,
- avoid silently forwarding arbitrary keywords.


# Problem 11 — Generate a frozen instance model from a class declaration

Now we will use a metaclass to generate small pieces of instance behavior.

A class will declare a tuple of field names:

```python
class Point(FrozenFields, fields=("x", "y")):
    pass
```

The metaclass will generate:

- `__slots__`,
- a constructor,
- iteration,
- a representation,
- and protection against adding undeclared attributes.

This example is intentionally "magical" so we can examine the trade-offs.


### Step 1 — Think carefully about `__slots__`

`__slots__` must exist when the class is created.

Attaching it after class creation does not retroactively change the instance layout.

So this is a genuine class-creation concern.


In [40]:
class FrozenFieldsMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        abstract=False,
        **kwargs,
    ):
        fields = tuple(fields)

        if not abstract:
            if not fields:
                raise TypeError(
                    f"{name} must declare at least one field"
                )

            if len(set(fields)) != len(fields):
                raise TypeError(
                    f"{name} contains duplicate field names"
                )

            for field in fields:
                if not isinstance(field, str) or not field.isidentifier():
                    raise TypeError(
                        f"Invalid field name: {field!r}"
                    )

        namespace["__field_names__"] = fields
        namespace["__slots__"] = fields

        if not abstract:
            def __init__(self, *args, **kwargs):
                if len(args) > len(fields):
                    raise TypeError("Too many positional arguments")

                values = dict(zip(fields, args))

                overlap = set(values) & set(kwargs)
                if overlap:
                    raise TypeError(
                        f"Multiple values for: {sorted(overlap)}"
                    )

                values.update(kwargs)

                unknown = set(values) - set(fields)
                if unknown:
                    raise TypeError(
                        f"Unknown field(s): {sorted(unknown)}"
                    )

                missing = [field for field in fields if field not in values]
                if missing:
                    raise TypeError(
                        f"Missing field(s): {missing}"
                    )

                for field in fields:
                    setattr(self, field, values[field])

            def __iter__(self):
                for field in fields:
                    yield getattr(self, field)

            def __repr__(self):
                body = ", ".join(
                    f"{field}={getattr(self, field)!r}"
                    for field in fields
                )
                return f"{type(self).__name__}({body})"

            namespace.setdefault("__init__", __init__)
            namespace.setdefault("__iter__", __iter__)
            namespace.setdefault("__repr__", __repr__)

        return super().__new__(mcls, name, bases, namespace)


### Step 2 — Create the root class and a concrete record


In [41]:
class FrozenFields(
    metaclass=FrozenFieldsMeta,
    abstract=True,
):
    __slots__ = ()


class Point(
    FrozenFields,
    fields=("x", "y"),
):
    pass


p = Point(10, y=20)

assert tuple(p) == (10, 20)
assert repr(p) == "Point(x=10, y=20)"


### Step 3 — Verify that arbitrary attributes are blocked

Because the generated class has slots and no instance dictionary, undeclared attributes cannot be added.


In [42]:
try:
    p.z = 30
except AttributeError as exc:
    print(exc)
else:
    raise AssertionError("Unexpected attribute should be blocked")


'Point' object has no attribute 'z' and no __dict__ for setting new attributes


### Step 4 — Compare this with `dataclasses`

This example demonstrates why metaclasses can be powerful, but Python already has a standard tool for this family of problems:

```python
@dataclass(slots=True)
class Point:
    x: int
    y: int
```

For ordinary data records, `dataclasses` are clearer, better known, and more tooling-friendly.

The metaclass is justified here mainly as a learning exercise about class layout.


# Problem 12 — Capstone: build a declarative state-machine DSL

We will combine several techniques into a small framework.

A class will declare transition methods like this:

```python
@transition("draft", "published")
def publish(self):
    ...
```

The metaclass will inspect those decorated methods and assemble a transition table.

The resulting class should be able to:

- report valid transitions,
- reject duplicate transitions,
- execute a transition by target state,
- inherit transitions,
- override inherited transitions.


### Step 1 — Make the decorator intentionally simple

The decorator should not mutate the class or register global state.

It only attaches metadata to the function object.


In [43]:
def transition(source, target):
    if not isinstance(source, str) or not source:
        raise TypeError("source must be a non-empty string")

    if not isinstance(target, str) or not target:
        raise TypeError("target must be a non-empty string")

    def decorator(func):
        func.__transition__ = (source, target)
        return func

    return decorator


### Step 2 — Decide the transition-table shape

We will map:

```python
(source_state, target_state) -> method_name
```

Storing method names instead of bound method objects keeps the class-level metadata simple and lets normal method binding happen later.


### Step 3 — Build the metaclass incrementally

First we copy inherited transitions.

Then we scan methods declared in the new class body.

A local transition with the same `(source, target)` pair replaces an inherited one.

But two methods in the *same* class body may not declare the same transition.


In [44]:
from types import MappingProxyType


class StateMachineMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        transitions = {}

        for base in bases:
            transitions.update(
                getattr(base, "__transitions__", {})
            )

        local_seen = set()

        for attr_name, value in namespace.items():
            spec = getattr(value, "__transition__", None)

            if spec is None:
                continue

            if spec in local_seen:
                raise TypeError(
                    f"{name} declares duplicate transition {spec}"
                )

            local_seen.add(spec)
            transitions[spec] = attr_name

        namespace["__transitions__"] = MappingProxyType(
            transitions
        )

        return super().__new__(mcls, name, bases, namespace)


### Step 4 — Create the reusable base class

The base class uses the metadata assembled by the metaclass.

Notice that ordinary runtime behavior stays in ordinary methods.


In [45]:
class StateMachine(metaclass=StateMachineMeta):
    initial_state = None

    def __init__(self):
        if self.initial_state is None:
            raise TypeError(
                f"{type(self).__name__} must define initial_state"
            )
        self.state = self.initial_state

    @classmethod
    def valid_transitions(cls):
        return tuple(cls.__transitions__)

    def can_transition_to(self, target):
        return (self.state, target) in self.__transitions__

    def transition_to(self, target):
        key = (self.state, target)

        if key not in self.__transitions__:
            raise ValueError(
                f"No transition from {self.state!r} to {target!r}"
            )

        method_name = self.__transitions__[key]
        method = getattr(self, method_name)

        result = method()
        self.state = target
        return result


### Step 5 — Define a concrete workflow

The class body now reads almost like a small domain-specific language.


In [46]:
class ArticleWorkflow(StateMachine):
    initial_state = "draft"

    @transition("draft", "review")
    def submit_for_review(self):
        return "submitted"

    @transition("review", "draft")
    def request_changes(self):
        return "changes requested"

    @transition("review", "published")
    def publish(self):
        return "published"


### Step 6 — Exercise the workflow one step at a time


In [47]:
article = ArticleWorkflow()

assert article.state == "draft"
assert article.can_transition_to("review")
assert not article.can_transition_to("published")

result = article.transition_to("review")

assert result == "submitted"
assert article.state == "review"
assert article.can_transition_to("published")


Now publish it.


In [48]:
result = article.transition_to("published")

assert result == "published"
assert article.state == "published"

print(article.state)


published


### Step 7 — Test an invalid transition


In [49]:
another = ArticleWorkflow()

try:
    another.transition_to("published")
except ValueError as exc:
    print(exc)
else:
    raise AssertionError(
        "draft -> published should not be allowed directly"
    )


No transition from 'draft' to 'published'


### Step 8 — Inherit and override a transition

A subclass may replace the method used for an inherited transition by declaring the same transition pair.


In [50]:
class AuditedArticleWorkflow(ArticleWorkflow):
    @transition("review", "published")
    def publish_with_audit(self):
        return "published with audit record"


audited = AuditedArticleWorkflow()
audited.transition_to("review")

result = audited.transition_to("published")

assert result == "published with audit record"
assert (
    AuditedArticleWorkflow.__transitions__[
        ("review", "published")
    ]
    == "publish_with_audit"
)

print(result)


published with audit record


### Step 9 — Why this is a reasonable metaclass use case

The transition table is a property of the **class definition**.

We need to inspect the entire completed class body, combine it with inherited metadata, and create class-level structure.

That is a defensible use of a metaclass.

Still, a class decorator could also implement this design. The choice depends on whether automatic inheritance behavior and framework-wide enforcement justify the metaclass.


# Problem 13 — Refactor a metaclass away when `__init_subclass__` is enough

Learning metaclasses includes learning when *not* to use them.

Suppose all we want is to register subclasses by a declared `kind`.

We do not need:

- a custom namespace,
- pre-creation transformation,
- special instance construction,
- custom `isinstance` behavior.

So `__init_subclass__` is enough.


### Step 1 — Implement the smaller solution directly


In [51]:
class Document:
    registry = {}

    def __init_subclass__(cls, *, kind=None, **kwargs):
        super().__init_subclass__(**kwargs)

        if kind is None:
            return

        if kind in cls.registry:
            raise TypeError(
                f"Duplicate document kind: {kind!r}"
            )

        cls.kind = kind
        cls.registry[kind] = cls


### Step 2 — Add subclasses

The class-header syntax remains declarative even though there is no custom metaclass.


In [52]:
class Invoice(Document, kind="invoice"):
    pass


class Receipt(Document, kind="receipt"):
    pass


assert Document.registry["invoice"] is Invoice
assert Document.registry["receipt"] is Receipt

print(Document.registry)


{'invoice': <class '__main__.Invoice'>, 'receipt': <class '__main__.Receipt'>}


### Step 3 — Compare the designs

A metaclass would work here, but it would add:

- another advanced concept,
- possible metaclass conflicts,
- more surprising inheritance behavior,
- more maintenance burden.

`__init_subclass__` reacts to subclass creation directly and expresses exactly what we need.


# Problem 14 — Compare a dynamic class factory with a metaclass

Metaclasses are not the only way to create classes programmatically.

Sometimes you genuinely want to build a class from data at runtime.

In that case, calling `type(name, bases, namespace)` directly may be simpler.


### Step 1 — Build a class factory

Our factory will create a simple record class with a fixed set of fields.


In [53]:
def make_record_class(name, fields):
    fields = tuple(fields)

    def __init__(self, **kwargs):
        unknown = set(kwargs) - set(fields)
        if unknown:
            raise TypeError(
                f"Unknown field(s): {sorted(unknown)}"
            )

        missing = set(fields) - set(kwargs)
        if missing:
            raise TypeError(
                f"Missing field(s): {sorted(missing)}"
            )

        for field in fields:
            setattr(self, field, kwargs[field])

    def as_tuple(self):
        return tuple(
            getattr(self, field)
            for field in fields
        )

    namespace = {
        "__fields__": fields,
        "__init__": __init__,
        "as_tuple": as_tuple,
    }

    return type(name, (), namespace)


### Step 2 — Create a class at runtime


In [54]:
Coordinate = make_record_class(
    "Coordinate",
    ("x", "y", "z"),
)

point = Coordinate(x=1, y=2, z=3)

assert point.as_tuple() == (1, 2, 3)
assert Coordinate.__name__ == "Coordinate"

print(Coordinate)
print(point.as_tuple())


<class '__main__.Coordinate'>
(1, 2, 3)


### Step 3 — Decide which tool matches the problem

Use a metaclass when users should write normal-looking class declarations and you need to customize how those declarations become classes.

Use a class factory when the program is itself constructing class definitions from runtime data.

They are related mechanisms, but they solve different interface problems.


# Mini-drills — Predict before running

These are short review problems.

Try to answer each question mentally before running the code.


### Drill 1

Will the metaclass method below run when the class is *defined*, when it is *instantiated*, or both?


In [55]:
drill_events = []

class DrillMeta(type):
    def __new__(mcls, name, bases, namespace):
        drill_events.append(("class-created", name))
        return super().__new__(mcls, name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        drill_events.append(("instance-created", cls.__name__))
        return super().__call__(*args, **kwargs)


class DrillClass(metaclass=DrillMeta):
    pass


before_instance = tuple(drill_events)
DrillClass()
after_instance = tuple(drill_events)

print("Before instance:", before_instance)
print("After instance:", after_instance)

assert before_instance == (
    ("class-created", "DrillClass"),
)
assert after_instance[-1] == (
    "instance-created",
    "DrillClass",
)


Before instance: (('class-created', 'DrillClass'),)
After instance: (('class-created', 'DrillClass'), ('instance-created', 'DrillClass'))


### Drill 2

If a descriptor is present in the namespace passed to `type.__new__`, Python calls its `__set_name__` automatically.

Let's verify it.


In [56]:
class NameAware:
    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name


class NameAwareMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace["injected"] = NameAware()
        return super().__new__(mcls, name, bases, namespace)


class InjectedDescriptorDemo(metaclass=NameAwareMeta):
    pass


assert InjectedDescriptorDemo.injected.name == "injected"
assert InjectedDescriptorDemo.injected.owner is InjectedDescriptorDemo

print(InjectedDescriptorDemo.injected.name)


injected


### Drill 3

Which mechanism is usually the simplest fit?

- Change one class explicitly → class decorator
- React whenever a subclass is created → `__init_subclass__`
- Manage one attribute → descriptor
- Control the class namespace while its body executes → metaclass `__prepare__`
- Control ordinary instance creation for an entire family of classes → metaclass `__call__`


# Best practices — A practical decision guide

Before writing a custom metaclass, ask these questions in order.

### 1. Can a normal base class solve it?

If yes, use ordinary inheritance.

### 2. Is the behavior about one attribute?

Use a descriptor or property.

### 3. Do I only need to react to subclasses?

Use `__init_subclass__`.

### 4. Do I only need to transform one class after creation?

Use a class decorator.

### 5. Is the problem primarily static typing?

Consider `Protocol`, `TypedDict`, generics, or normal type annotations.

### 6. Do I genuinely need to control class creation?

Now a metaclass may be appropriate.


## Metaclass implementation checklist

When a metaclass really is justified:

- use `super()` cooperatively,
- preserve special namespace entries such as `__classcell__`,
- validate class-header keywords explicitly,
- keep error messages specific,
- avoid silently replacing user-defined methods,
- keep instance-level logic outside the metaclass when possible,
- test inheritance,
- test multiple inheritance,
- test interaction with `ABCMeta` or framework metaclasses,
- understand descriptor timing and `__set_name__`,
- document generated methods and attributes,
- be cautious when overriding metaclass `__call__`,
- prefer readable declarative APIs over hidden magic.


# Final synthesis

The most important mental distinction is:

```text
ordinary class methods
    -> behavior of instances

class methods / static methods
    -> behavior exposed through the class

descriptors
    -> behavior of attributes

__init_subclass__
    -> reaction to subclass creation

class decorators
    -> explicit post-processing of a class

metaclasses
    -> creation and class-level behavior of classes themselves
```

Metaclasses are not "more object-oriented" than the alternatives.

They are simply one hook in Python's object model — useful when the problem truly lives at the class-creation level.


In [57]:
# Final notebook smoke test

assert type(MarkedBase) is MarkingMeta
assert TutorialExample.__assignment_order__[0] == "first"
assert RetryPolicy.attempts == 3
assert SafeChild().message() == "parent -> safe child"
assert UserRecord(1, "grace").username == "grace"
assert issubclass(SimpleSerializer, SerializerLike)
assert Tag("PYTHON") is Tag("python")
assert CommandMeta.registry["ping"] is PingCommand
assert UsersResource.__version__ == 2
assert tuple(Point(1, 2)) == (1, 2)
assert ("review", "published") in ArticleWorkflow.__transitions__
assert Document.registry["invoice"] is Invoice
assert Coordinate(x=1, y=2, z=3).as_tuple() == (1, 2, 3)

print("All tutorial examples passed.")


All tutorial examples passed.
